# Question 4

In [ ]:
from abc import ABC
from abc import abstractmethod
import networkx as nx
import numpy as np
import math
from tqdm import tqdm

class LinkPrediction(ABC):
    def __init__(self, graph):
        """
        Constructor
        
        Parameters
        ----------
        graph : Networkx graph
        """
        self.graph = graph
        self.N = len(graph)

    def neighbors(self, v):
        """
        Return the neighbors list of a node
        """
        neighbors_list = self.graph.neighbors(v)
        return list(neighbors_list)

    @abstractmethod
    def fit(self, pairs):
        raise NotImplementedError("Fit must be implemented")

class CommonNeighbors(LinkPrediction):
    def __init__(self, graph):
        super(CommonNeighbors, self).__init__(graph)

    def fit(self, pairs):
        """
        Calcule le nombre de voisins communs pour chaque paire.
        """
        scores = []
        for u, v in tqdm(pairs, desc="Common Neighbors"):
            set_u = set(self.neighbors(u))
            set_v = set(self.neighbors(v))
            scores.append(len(set_u.intersection(set_v)))
        return np.array(scores)

class Jaccard(LinkPrediction):
    def __init__(self, graph):
        super(Jaccard, self).__init__(graph)

    def fit(self, pairs):
        """
        Calcule le coefficient de Jaccard.
        """
        scores = []
        for u, v in tqdm(pairs, desc="Jaccard"):
            set_u = set(self.neighbors(u))
            set_v = set(self.neighbors(v))
            intersection = len(set_u.intersection(set_v))
            union = len(set_u.union(set_v))
            scores.append(intersection / union if union > 0 else 0)
        return np.array(scores)

class AdamicAdar(LinkPrediction):
    def __init__(self, graph):
        super(AdamicAdar, self).__init__(graph)

    def fit(self, pairs):
        """
        Calcule l'indice Adamic/Adar.
        """
        scores = []
        for u, v in tqdm(pairs, desc="Adamic/Adar"):
            set_u = set(self.neighbors(u))
            set_v = set(self.neighbors(v))
            common_neighbors = set_u.intersection(set_v)
            
            total = 0
            for z in common_neighbors:
                degree = self.graph.degree(z)
                if degree > 1:
                    total += 1.0 / math.log10(degree)
            scores.append(total)
        return np.array(scores)


In [ ]:
import random as rd
from tqdm import tqdm


def remove_edges(G, fraction):
    G_copy = G.copy()
    edges = list(G_copy.edges())
    num_to_remove = int(len(edges) * fraction)

    edges_removed = rd.sample(edges, num_to_remove)
    G_copy.remove_edges_from(edges_removed)
    return G_copy, edges_removed



def eval_link_prediction(model, G, edges_removed, k_values=[50, 100, 200, 400]):
    nodes = list(G.nodes())
    pairs = []

    for i in tqdm(range(len(nodes)), desc="Génération des paires candidates"):
        for j in range(i + 1, len(nodes)):
            u, v = nodes[i], nodes[j]
            if not G.has_edge(u, v):
                pairs.append((u, v))

    print(f"Fin de la génération des paires candidates, nombre de candidats: {len(pairs)}")
    scores = model.fit(pairs)
    print(f"Tri des prédictions...")
    predictions = sorted(zip(pairs, scores), key=lambda x: x[1], reverse=True)
    edges_removed_set = {tuple(sorted(edge)) for edge in edges_removed}
    print(f"Fin tri des prédictions, évaluation des top-k...")

    results = {}
    for k in k_values:
        top_k_predicted = [tuple(sorted(pair)) for pair, score in predictions[:k]]

        tp = 0
        for pred in top_k_predicted:
            if pred in edges_removed_set:
                tp += 1

        precision_at_k = tp / k
        recall_at_k = tp / len(edges_removed)
        top_k_rate = (tp / k) * 100

        results[k] = {
            'precision': precision_at_k,
            'recall': recall_at_k,
            'top@k_rate': top_k_rate
        }

    return results


In [3]:
graphs_used = ["data/MIT8.gml", "data/UPenn7.gml"]
f_values = [0.05, 0.1, 0.15, 0.2]
k_values = [50, 100, 200, 400]
predictors = {
    "Common Neighbors": CommonNeighbors,
    "Jaccard": Jaccard,
    "Adamic/Adar": AdamicAdar,
}

def use_graph(graph_file):
    G = nx.read_gml(graph_file)
    print(f"\nGraphe: {graph_file}")

    for f in f_values:
        G_copy, e_removed = remove_edges(G, f)
        print(f"Fraction d'arêtes supprimées: {f}")

        summary = {}
        for predictor_name, predictor_class in predictors.items():
            model = predictor_class(G_copy)
            resultats = eval_link_prediction(model, G_copy, e_removed, k_values)
            avg_precision = sum(resultats[k]['precision'] for k in k_values) / len(k_values)
            avg_recall = sum(resultats[k]['recall'] for k in k_values) / len(k_values)
            summary[predictor_name] = (avg_precision, avg_recall)
            print(f"{predictor_name}: {resultats}")

        best_predictor = max(summary, key=lambda name: summary[name][0])
        best_precision, best_recall = summary[best_predictor]
        print(
            f"Meilleurs prédictueurs de {predictor_name} pour f={f} : {best_predictor} "
            f"(précision moyenne={best_precision:.4f}, recall moyen={best_recall:.4f})"
        )


In [4]:
use_graph("data/MIT8.gml")


Graphe: data/MIT8.gml
Fraction d'arêtes supprimées: 0.05


Génération des paires candidates: 100%|██████████| 6440/6440 [00:03<00:00, 1994.55it/s]


Fin de la génération des paires candidates, nombre de candidats: 20494890


Common Neighbors: 100%|██████████| 20494890/20494890 [01:56<00:00, 175303.70it/s]


Fin de l'ajustement du modèle, tri des prédictions...
Fin du tri des prédictions, évaluation des top-k...
Common Neighbors: {50: {'precision': 0.68, 'recall': 0.0027065753860850182, 'top@k_rate': 68.0}, 100: {'precision': 0.66, 'recall': 0.005253940455341506, 'top@k_rate': 66.0}, 200: {'precision': 0.51, 'recall': 0.008119726158255056, 'top@k_rate': 51.0}, 400: {'precision': 0.4525, 'recall': 0.01440853367298201, 'top@k_rate': 45.25}}


...

Fin de l'ajustement du modèle, tri des prédictions...
Fin du tri des prédictions, évaluation des top-k...
Jaccard: {50: {'precision': 0.24, 'recall': 0.00023880597014925374, 'top@k_rate': 24.0}, 100: {'precision': 0.18, 'recall': 0.0003582089552238806, 'top@k_rate': 18.0}, 200: {'precision': 0.325, 'recall': 0.0012935323383084577, 'top@k_rate': 32.5}, 400: {'precision': 0.42, 'recall': 0.003343283582089552, 'top@k_rate': 42.0}}


Génération des paires candidates: 100%|██████████| 6440/6440 [00:03<00:00, 1931.08it/s]


Fin de la génération des paires candidates, nombre de candidats: 20532578


Adamic/Adar: 100%|██████████| 20532578/20532578 [01:55<00:00, 178210.91it/s]


Fin de l'ajustement du modèle, tri des prédictions...
Fin du tri des prédictions, évaluation des top-k...
Adamic/Adar: {50: {'precision': 0.86, 'recall': 0.0008557213930348259, 'top@k_rate': 86.0}, 100: {'precision': 0.88, 'recall': 0.0017512437810945274, 'top@k_rate': 88.0}, 200: {'precision': 0.87, 'recall': 0.0034626865671641793, 'top@k_rate': 87.0}, 400: {'precision': 0.8525, 'recall': 0.006786069651741293, 'top@k_rate': 85.25}}
Meilleurs prédictueurs de Adamic/Adar pour f=0.2 : Adamic/Adar (précision moyenne=0.8656, recall moyen=0.0032)


In [4]:
use_graph("data/Caltech36.gml")


Graphe: data/Caltech36.gml
Fraction d'arêtes supprimées: 0.05


Génération des paires candidates: 100%|██████████| 769/769 [00:00<00:00, 13651.99it/s]


Fin de la génération des paires candidates, nombre de candidats: 279472


Common Neighbors: 100%|██████████| 279472/279472 [00:00<00:00, 306590.18it/s]


Fin de l'ajustement du modèle, tri des prédictions...
Fin du tri des prédictions, évaluation des top-k...
Common Neighbors: {50: {'precision': 0.46, 'recall': 0.027644230769230768, 'top@k_rate': 46.0}, 100: {'precision': 0.39, 'recall': 0.046875, 'top@k_rate': 39.0}, 200: {'precision': 0.345, 'recall': 0.0829326923076923, 'top@k_rate': 34.5}, 400: {'precision': 0.2825, 'recall': 0.13581730769230768, 'top@k_rate': 28.249999999999996}}


...

Fin de l'ajustement du modèle, tri des prédictions...
Fin du tri des prédictions, évaluation des top-k...
Jaccard: {50: {'precision': 0.44, 'recall': 0.006604623236265386, 'top@k_rate': 44.0}, 100: {'precision': 0.58, 'recall': 0.01741218853197238, 'top@k_rate': 57.99999999999999}, 200: {'precision': 0.505, 'recall': 0.03032122485740018, 'top@k_rate': 50.5}, 400: {'precision': 0.51, 'recall': 0.06124287000900631, 'top@k_rate': 51.0}}


Génération des paires candidates: 100%|██████████| 769/769 [00:00<00:00, 14739.18it/s]


Fin de la génération des paires candidates, nombre de candidats: 281971


Adamic/Adar: 100%|██████████| 281971/281971 [00:01<00:00, 268421.13it/s]


Fin de l'ajustement du modèle, tri des prédictions...
Fin du tri des prédictions, évaluation des top-k...
Adamic/Adar: {50: {'precision': 0.88, 'recall': 0.013209246472530772, 'top@k_rate': 88.0}, 100: {'precision': 0.82, 'recall': 0.02461723206244371, 'top@k_rate': 82.0}, 200: {'precision': 0.745, 'recall': 0.04473131191834284, 'top@k_rate': 74.5}, 400: {'precision': 0.6625, 'recall': 0.0795556889822876, 'top@k_rate': 66.25}}
Meilleurs prédictueurs de Adamic/Adar pour f=0.2 : Adamic/Adar (précision moyenne=0.7769, recall moyen=0.0405)
